# Final Model Training & Competition Submission

This notebook retrains the best model (SVR, tuned) on ALL of train.csv and generates predictions on the competition test set.

The best hyperparameters were found during the model comparison and tuning process documented in regression_process_report.ipynb.

For the submission, I do not cap the target — capping prevents predicting high earners correctly on unseen data. Instead, I train on the full uncapped log-transformed target.

In [ ]:
import os

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_regression, VarianceThreshold
from sklearn.linear_model import LinearRegression as LR_imp, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor, StackingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

print("Imports done.")

Imports done.


## Best Model Hyperparameters (from tuning in regression_process_report.ipynb)

SVR (tuned) was the best model by validation RMSE ($77,769). Parameters found via RandomizedSearchCV (200 iterations, 5-fold CV).

In [7]:
# Best hyperparameters found during tuning
best_svr_params = {
    'C': 9.38761887501214,
    'epsilon': 0.30569513623318023,
    'gamma': 0.002661557290464444
}

## Load & Clean Data

In [8]:
# Reload and clean full training data (only remove clear errors)
df_full = pd.read_csv("data/train.csv")
df_full = df_full[df_full['annual.pay.usd'] >= 1000].copy()

y_full = df_full['annual.pay.usd']
X_full = df_full.drop(columns=['annual.pay.usd'])

# Log-transform target
y_full_log = np.log1p(y_full)

# Load competition test set
df_test = pd.read_csv("data/test.csv")
test_ids = df_test['id']
X_comp_test = df_test.drop(columns=['id'])

print(f"Full training: {X_full.shape[0]} rows")
print(f"Competition test: {X_comp_test.shape[0]} rows")

Full training: 2362 rows
Competition test: 628 rows


## Preprocessing Pipeline

I apply the exact same preprocessing pipeline used during training to both the full training set and the competition test set. This ensures consistency and the model benefits from more training data.

In [9]:
# Domain knowledge column filter 
drop_cols = ['first.help.source', 'personal.os', 'comm.tools', 'dev.environments',
             'ai.sentiment', 'ai.trust', 'ai.complex.rating', 'ai.job.threat',
             'how.learned.coding', 'side.coding', 'job.satisfaction',
             'cloud.hosting', 'daily.search.time', 'people.manager']
X_full = X_full.drop(columns=[c for c in drop_cols if c in X_full.columns])
X_comp_test = X_comp_test.drop(columns=[c for c in drop_cols if c in X_comp_test.columns])

# MissingIndicators for high-missing salary-relevant columns
for df_x in [X_full, X_comp_test]:
    df_x['experience_missing'] = df_x['experience.years'].isnull().astype(int)
    df_x['industry_missing'] = df_x['industry'].isnull().astype(int)
    df_x['daily_answer_missing'] = df_x['daily.answer.time'].isnull().astype(int)

# Handling industry: fill NaN with "Missing" 
for df_x in [X_full, X_comp_test]:
    df_x['industry'] = df_x['industry'].fillna('Missing')

# Handling daily.answer.time: ordinal with -1 for missing 
time_order = {"Less than 15 minutes a day": 0, "15-30 minutes a day": 1,
              "30-60 minutes a day": 2, "60-120 minutes a day": 3, "Over 120 minutes a day": 4}
for df_x in [X_full, X_comp_test]:
    df_x['daily.answer.time'] = df_x['daily.answer.time'].map(time_order).fillna(-1)

# Handling experience.years: impute from coding.years.professional + coding.years.total 
exp_predictors_sub = ['coding.years.professional', 'coding.years.total']
mask_exp_full = (X_full['experience.years'].notna() &
                 X_full['coding.years.professional'].notna() &
                 X_full['coding.years.total'].notna())
exp_imp_sub = LR_imp()
exp_imp_sub.fit(X_full.loc[mask_exp_full, exp_predictors_sub],
                X_full.loc[mask_exp_full, 'experience.years'])
for df_x in [X_full, X_comp_test]:
    missing_exp = (df_x['experience.years'].isnull() &
                   df_x['coding.years.professional'].notna() &
                   df_x['coding.years.total'].notna())
    if missing_exp.any():
        df_x.loc[missing_exp, 'experience.years'] = exp_imp_sub.predict(
            df_x.loc[missing_exp, exp_predictors_sub])
    still_missing = df_x['experience.years'].isnull()
    if still_missing.any():
        df_x.loc[still_missing, 'experience.years'] = X_full.loc[mask_exp_full, 'experience.years'].median()

# REGION TARGET ENCODING 
region_full = X_full['region'].copy()
region_comp = X_comp_test['region'].copy()

global_mean_sub = y_full_log.mean()
region_enc_full = pd.Series(index=X_full.index, dtype=float)
kf_sub = KFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, vl_idx in kf_sub.split(X_full):
    fold_means = y_full_log.iloc[tr_idx].groupby(region_full.iloc[tr_idx]).mean()
    region_enc_full.iloc[vl_idx] = region_full.iloc[vl_idx].map(fold_means).fillna(global_mean_sub)

full_region_means_sub = y_full_log.groupby(region_full).mean()
region_enc_comp = region_comp.map(full_region_means_sub).fillna(global_mean_sub)

X_full['region_target_enc'] = region_enc_full.values
X_comp_test['region_target_enc'] = region_enc_comp.values
X_full = X_full.drop(columns=['region'])
X_comp_test = X_comp_test.drop(columns=['region'])

# Ordinal encoding 
age_order = {"18-24": 0, "25-34": 1, "35-44": 2, "45-54": 3, "55+": 4}
edu_order = {
    "Primary/elementary school": 0,
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
    "Some college/university study without earning a degree": 2,
    "Associate degree (A.A., A.S., etc.)": 3,
    "Bachelor'ArithmeticErrors degree (B.A., B.S., B.Eng., etc.)": 4,
    "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 6,
    "Something else": 2
}
company_order = {
    "Just me - I am a freelancer, sole proprietor, etc.": 0,
    "2 to 9 employees": 1, "10 to 19 employees": 2,
    "20 to 99 employees": 3, "100 to 499 employees": 4,
    "500 to 999 employees": 5, "1,000 to 4,999 employees": 6,
    "5,000 to 9,999 employees": 7, "10,000 or more employees": 8,
    "I don't know": np.nan
}
influence_order = {"I have little or no influence": 0, "I have some influence": 1, "I have a great deal of influence": 2}

for df_x in [X_full, X_comp_test]:
    df_x["age.group"] = df_x["age.group"].map(age_order)
    df_x["education"] = df_x["education"].map(edu_order)
    df_x["company.size"] = df_x["company.size"].map(company_order)
    df_x["tech.purchase.influence"] = df_x["tech.purchase.influence"].map(influence_order)

# Binary encoding
for df_x in [X_full, X_comp_test]:
    df_x["is_dev"] = (df_x["is.dev.professional"] == "I am a developer by profession").astype(int)
    df_x.drop(columns=["is.dev.professional"], inplace=True)

# Multi-select columns — count only (number of items selected) 
count_only_cols = ['prog.languages', 'databases', 'cloud.platforms',
                   'web.frameworks', 'other.tech', 'dev.tools',
                   'work.os', 'project.mgmt.tools', 'ai.search.tools', 'ai.tools.used']

for col in count_only_cols:
    if col not in X_full.columns:
        continue
    for df_x in [X_full, X_comp_test]:
        df_x[f"{col}__count"] = df_x[col].fillna('').str.split(';').apply(lambda x: len([i for i in x if i.strip()]))
        df_x.drop(columns=[col], inplace=True)

# dev.role one-hot
dev_dummies_full = pd.get_dummies(X_full['dev.role'], prefix='dev_role')
dev_dummies_test = pd.get_dummies(X_comp_test['dev.role'], prefix='dev_role')
dev_dummies_test = dev_dummies_test.reindex(columns=dev_dummies_full.columns, fill_value=0)
X_full = pd.concat([X_full.drop(columns=['dev.role']), dev_dummies_full], axis=1)
X_comp_test = pd.concat([X_comp_test.drop(columns=['dev.role']), dev_dummies_test], axis=1)

# Nominal one-hot (including industry with "Missing" category) 
nominal_cols = ["employment.type", "work.location", "build.vs.buy", "uses.ai", "industry"]
nominal_cols = [c for c in nominal_cols if c in X_full.columns]
X_full = pd.get_dummies(X_full, columns=nominal_cols, drop_first=True)
X_comp_test = pd.get_dummies(X_comp_test, columns=nominal_cols, drop_first=True)
X_comp_test = X_comp_test.reindex(columns=X_full.columns, fill_value=0)

# Missing value handling: drop rows from training, KNNImputer for test 
print(f"Full train before dropping NaN: {X_full.shape[0]} rows")
mask_full_complete = ~X_full.isnull().any(axis=1)
X_full = X_full[mask_full_complete].copy()
y_full_log = y_full_log[mask_full_complete].copy()
print(f"Full train after dropping NaN: {X_full.shape[0]} rows")

# Clean up column names
clean = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in X_full.columns]
X_full.columns = clean
X_comp_test.columns = clean

# KNNImputer for competition test
numeric_features_sub = ["coding_years_total", "coding_years_professional", "experience_years"]
ordinal_features_sub = ["age_group", "education", "company_size", "tech_purchase_influence", "daily_answer_time"]
impute_cols_sub = [c for c in numeric_features_sub + ordinal_features_sub if c in X_comp_test.columns and X_comp_test[c].isnull().any()]
if impute_cols_sub:
    knn_imp_sub = KNNImputer(n_neighbors=5)
    knn_imp_sub.fit(X_full[impute_cols_sub])
    X_comp_test[impute_cols_sub] = knn_imp_sub.transform(X_comp_test[impute_cols_sub])
X_comp_test = X_comp_test.fillna(0)

# Winsorization 
numeric_features_clean = ["coding_years_total", "coding_years_professional", "experience_years"]
for col in numeric_features_clean:
    if col in X_full.columns:
        p1 = X_full[col].quantile(0.01)
        p99 = X_full[col].quantile(0.99)
        X_full[col] = X_full[col].clip(lower=p1, upper=p99)
        X_comp_test[col] = X_comp_test[col].clip(lower=p1, upper=p99)

# Interaction features
for df_x in [X_full, X_comp_test]:
    df_x["exp_professional_sq"] = df_x["coding_years_professional"] ** 2
    df_x["exp_years_sq"] = df_x["experience_years"] ** 2
    df_x["exp_x_company"] = df_x["coding_years_professional"] * df_x["company_size"]
    df_x["exp_x_education"] = df_x["coding_years_professional"] * df_x["education"]
    df_x["total_coding_x_company"] = df_x["coding_years_total"] * df_x["company_size"]
    df_x["exp_x_region"] = df_x["coding_years_professional"] * df_x["region_target_enc"]
    df_x["education_x_region"] = df_x["education"] * df_x["region_target_enc"]
    df_x["company_x_region"] = df_x["company_size"] * df_x["region_target_enc"]
    df_x["expyears_x_region"] = df_x["experience_years"] * df_x["region_target_enc"]

# Feature selection 
vt_full = VarianceThreshold(threshold=0.01)
vt_full.fit(X_full)
mask_vt_full = vt_full.get_support()
X_full = X_full.loc[:, mask_vt_full]
X_comp_test = X_comp_test.loc[:, mask_vt_full]

# Mutual information filter — keep top 40
mi_full = mutual_info_regression(X_full, y_full_log, random_state=42, n_neighbors=5)
mi_df_full = pd.DataFrame({"feature": X_full.columns, "mi_score": mi_full}).sort_values("mi_score", ascending=False)

keep_feats_sub = mi_df_full.head(40)["feature"].tolist()

X_comp_test = X_comp_test[keep_feats_sub]
X_full = X_full[keep_feats_sub]

print(f"Final shapes — X_full: {X_full.shape}, X_comp_test: {X_comp_test.shape}")

Full train before dropping NaN: 2362 rows
Full train after dropping NaN: 555 rows
Final shapes — X_full: (555, 40), X_comp_test: (628, 40)


## Train Final Model & Generate Submission

I train the best SVR model on all available training data and generate predictions on the competition test set. Predictions are transformed back to original dollar scale via expm1 (inverse of log1p). Any negative predictions are clipped to $300 (minimum realistic salary).

In [ ]:
# Train SVR (best model) on all data and generate submission
submission_model = Pipeline([
    ("scaler", RobustScaler()),
    ("model", SVR(kernel="rbf", **best_svr_params))
])

# Train on all data
submission_model.fit(X_full, y_full_log)

# Predict on competition test set
preds_log = submission_model.predict(X_comp_test)
preds = np.expm1(preds_log)

# Clip any negative predictions to a reasonable minimum
preds = np.clip(preds, 300, None)

# Create submission file
submission = pd.DataFrame({'id': test_ids, 'annual.pay.usd': preds})
submission.to_csv('data/submission_v2.csv', index=False)

print(f"Submission saved: data/submission_v2.csv")
print(f"Rows: {len(submission)}")
print(f"Prediction stats:")
print(f"  Mean: ${preds.mean():,.0f}")
print(f"  Median: ${np.median(preds):,.0f}")
print(f"  Min: ${preds.min():,.0f}")
print(f"  Max: ${preds.max():,.0f}")
print(f"\nModel used: SVR (RBF kernel)")
print(f"  C={best_svr_params['C']:.4f}, epsilon={best_svr_params['epsilon']:.4f}, gamma={best_svr_params['gamma']:.6f}")

submission.head(10)

Submission saved: data/submission_v2.csv
Rows: 628
Prediction stats:
  Mean: $40,070
  Median: $35,561
  Min: $7,829
  Max: $126,793

Model used: SVR (RBF kernel)
  C=9.3876, epsilon=0.3057, gamma=0.002662


,id,annual.pay.usd
0,0,18334.047898
1,1,39842.176337
2,2,48401.011792
3,3,58850.660051
4,4,22412.554251
5,5,87835.109747
6,6,34223.665009
7,7,68764.383712
8,8,21942.242793
9,9,49389.956437
